# Interpretable AI Fraud & Transaction Risk Intelligence

**A portfolio-grade fraud decision system built around human-readable transaction features — no anonymous V1–V28 PCA columns.**

This version uses the **Sparkov simulated credit-card transaction dataset**, whose fields resemble information a fraud team can actually interpret.

### Interpretable predictive signals
- transaction amount and log amount;
- merchant category;
- transaction hour, weekday, weekend and night flags;
- cardholder-to-merchant distance;
- city population;
- prior customer transaction count and prior average spend;
- amount versus the customer's prior average;
- time since the customer's previous transaction;
- prior merchant/category activity.

### Privacy and fairness choices
Raw names, card numbers, addresses, transaction IDs and exact raw coordinates are not model inputs. Gender and age are also excluded from prediction.

### Modelling
- class-weighted Logistic Regression baseline;
- XGBoost rare-event classifier;
- chronological train / validation / test split;
- PR-AUC-first evaluation;
- validation-only cost threshold optimisation;
- capture@K for investigator capacity;
- readable XGBoost SHAP contributions;
- ranked analyst investigation queue.

> Sparkov contains simulated transactions, not real customer records.

In [ ]:
# Environment setup
import sys, subprocess, importlib.util
if importlib.util.find_spec("xgboost") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
print("Environment ready.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix, precision_recall_curve,
    classification_report
)
from xgboost import XGBClassifier, DMatrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Libraries imported.")

## 1. Load interpretable Sparkov transactions

For reproducibility, the notebook streams a public GitHub-hosted copy of the Sparkov fraud-training file. To keep execution practical, it uses the first **350,000 chronologically ordered rows**.

In [ ]:
DATA_URL = "https://media.githubusercontent.com/media/Syed313110/Fraud_Detection_Hackathon/main/data/raw/fraudTrain.csv"
MAX_ROWS = 350_000

df = pd.read_csv(DATA_URL, nrows=MAX_ROWS, low_memory=False)
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()

required = {
    "trans_date_trans_time","cc_num","merchant","category","amt",
    "city_pop","lat","long","merch_lat","merch_long","is_fraud"
}
missing = sorted(required - set(df.columns))
if missing:
    raise RuntimeError(f"Dataset is missing required columns: {missing}")

df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
df = df.dropna(subset=["trans_date_trans_time","is_fraud","amt"]).copy()
df["is_fraud"] = pd.to_numeric(df["is_fraud"], errors="coerce").fillna(0).astype(int)
df = df.sort_values("trans_date_trans_time").reset_index(drop=True)

print(f"Rows loaded: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Fraud cases: {int(df['is_fraud'].sum()):,}")
print(f"Fraud rate: {df['is_fraud'].mean():.3%}")
print(f"Period: {df['trans_date_trans_time'].min()} -> {df['trans_date_trans_time'].max()}")
display(df[["trans_date_trans_time","merchant","category","amt","city_pop","is_fraud"]].head())

## 2. Leakage-safe, readable feature engineering

Behavioural features use only information available at or before each transaction. Customer/merchant IDs are used temporarily to construct history features and are then excluded from the model.

In [ ]:
print(f"Missing values before engineering: {int(df.isna().sum().sum()):,}")
print(f"Exact duplicate rows: {int(df.duplicated().sum()):,}")

df["hour"] = df["trans_date_trans_time"].dt.hour.astype(int)
df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek.astype(int)
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 4)).astype(int)
df["amount_log"] = np.log1p(df["amt"].clip(lower=0))
df["city_pop_log"] = np.log1p(pd.to_numeric(df["city_pop"], errors="coerce").fillna(0).clip(lower=0))

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2-lat1)
    dl = np.radians(lon2-lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(np.clip(a,0,1)))

df["distance_km"] = haversine_km(
    pd.to_numeric(df["lat"], errors="coerce").fillna(0).to_numpy(),
    pd.to_numeric(df["long"], errors="coerce").fillna(0).to_numpy(),
    pd.to_numeric(df["merch_lat"], errors="coerce").fillna(0).to_numpy(),
    pd.to_numeric(df["merch_long"], errors="coerce").fillna(0).to_numpy()
)

customer = df.groupby("cc_num", sort=False)
df["customer_txn_count_prior"] = customer.cumcount()
prior_spend = customer["amt"].cumsum() - df["amt"]
count_prior = df["customer_txn_count_prior"].replace(0, np.nan)
global_median = float(df["amt"].median())
df["customer_avg_amount_prior"] = (prior_spend/count_prior).fillna(global_median)
df["amount_vs_customer_avg"] = (
    df["amt"]/(df["customer_avg_amount_prior"].abs()+1.0)
).clip(0,100)
df["hours_since_prev_txn"] = (
    customer["trans_date_trans_time"].diff().dt.total_seconds()/3600
).fillna(999).clip(0,999)
df["merchant_prior_count"] = df.groupby("merchant", sort=False).cumcount()
df["category_prior_count"] = df.groupby("category", sort=False).cumcount()

preview = [
    "amt","category","hour","distance_km","customer_txn_count_prior",
    "customer_avg_amount_prior","amount_vs_customer_avg",
    "hours_since_prev_txn","merchant_prior_count","is_fraud"
]
display(df[preview].head(10))
print("PII/raw identifiers are NOT predictive features.")

## 3. Fraud patterns an interviewer can understand

In [ ]:
counts = df["is_fraud"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7,4.2))
ax.bar(["Legitimate","Fraud"], counts.values)
ax.set_title("Class imbalance")
ax.set_ylabel("Transactions")
for i,v in enumerate(counts.values):
    ax.text(i,v,f"{v:,}\n({v/len(df):.3%})",ha="center",va="bottom")
plt.tight_layout(); plt.show()

print(f"Always predicting legitimate would score {1-df['is_fraud'].mean():.3%} accuracy and catch zero fraud.")

category_stats = (
    df.groupby("category")
      .agg(transactions=("is_fraud","size"),frauds=("is_fraud","sum"),fraud_rate=("is_fraud","mean"))
      .sort_values("fraud_rate",ascending=False)
)
display(category_stats)

fig, ax = plt.subplots(figsize=(10,5))
category_stats.sort_values("fraud_rate")["fraud_rate"].mul(100).plot(kind="barh",ax=ax)
ax.set_title("Fraud rate by merchant category")
ax.set_xlabel("Fraud rate (%)")
plt.tight_layout(); plt.show()

hour_stats = df.groupby("hour")["is_fraud"].agg(["size","sum","mean"])
fig, ax = plt.subplots(figsize=(9,4.5))
ax.plot(hour_stats.index,hour_stats["mean"]*100,marker="o")
ax.set_title("Fraud rate by transaction hour")
ax.set_xlabel("Hour"); ax.set_ylabel("Fraud rate (%)"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 4. Chronological train / validation / test design

The first 60% of transactions train the model, the next 20% select the model/threshold, and the final 20% remain untouched until final evaluation.

In [ ]:
numeric_features = [
    "amt","amount_log","distance_km","city_pop_log","hour","day_of_week",
    "is_weekend","is_night","customer_txn_count_prior","customer_avg_amount_prior",
    "amount_vs_customer_avg","hours_since_prev_txn","merchant_prior_count","category_prior_count"
]
categorical_features = ["category"]
model_features = numeric_features + categorical_features

n = len(df)
train_end, val_end = int(n*0.60), int(n*0.80)
train_df, val_df, test_df = df.iloc[:train_end].copy(), df.iloc[train_end:val_end].copy(), df.iloc[val_end:].copy()

X_train, y_train = train_df[model_features], train_df["is_fraud"]
X_val, y_val = val_df[model_features], val_df["is_fraud"]
X_test, y_test = test_df[model_features], test_df["is_fraud"]

split_summary = pd.DataFrame({
    "rows":[len(train_df),len(val_df),len(test_df)],
    "frauds":[int(y_train.sum()),int(y_val.sum()),int(y_test.sum())],
    "fraud_rate":[y_train.mean(),y_val.mean(),y_test.mean()],
    "start":[train_df["trans_date_trans_time"].min(),val_df["trans_date_trans_time"].min(),test_df["trans_date_trans_time"].min()],
    "end":[train_df["trans_date_trans_time"].max(),val_df["trans_date_trans_time"].max(),test_df["trans_date_trans_time"].max()]
},index=["train","validation","test"])
display(split_summary)

preprocessor = ColumnTransformer([
    ("num",StandardScaler(with_mean=False),numeric_features),
    ("cat",OneHotEncoder(handle_unknown="ignore",min_frequency=10),categorical_features)
],sparse_threshold=1.0)

X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

feature_names = np.array([
    x.replace("num__","").replace("cat__","")
    for x in preprocessor.get_feature_names_out()
])
print(f"Train matrix: {X_train_t.shape}; readable transformed features: {len(feature_names)}")

## 5. Train Logistic Regression + XGBoost

In [ ]:
neg,pos = np.bincount(y_train.astype(int))
scale_pos_weight = neg/max(pos,1)

logit = LogisticRegression(
    class_weight="balanced",max_iter=800,solver="saga",
    random_state=RANDOM_STATE,n_jobs=-1
)
logit.fit(X_train_t,y_train)
val_prob_logit = logit.predict_proba(X_val_t)[:,1]

xgb_model = XGBClassifier(
    n_estimators=350,max_depth=5,learning_rate=0.05,
    subsample=0.85,colsample_bytree=0.85,min_child_weight=2,
    reg_lambda=3.0,reg_alpha=0.05,objective="binary:logistic",
    eval_metric="aucpr",tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,n_jobs=-1
)
xgb_model.fit(X_train_t,y_train)
val_prob_xgb = xgb_model.predict_proba(X_val_t)[:,1]

print(f"XGBoost fraud class weight: {scale_pos_weight:.1f}")
print("Models fitted.")

## 6. Validation model comparison — PR-AUC first

In [ ]:
def metrics_row(name,y_true,prob,threshold=0.5):
    pred = (np.asarray(prob)>=threshold).astype(int)
    return {
        "model":name,
        "PR_AUC":average_precision_score(y_true,prob),
        "ROC_AUC":roc_auc_score(y_true,prob),
        "precision":precision_score(y_true,pred,zero_division=0),
        "recall":recall_score(y_true,pred,zero_division=0),
        "F1":f1_score(y_true,pred,zero_division=0),
        "alerts":int(pred.sum())
    }

val_results = pd.DataFrame([
    metrics_row("Logistic Regression",y_val,val_prob_logit),
    metrics_row("XGBoost",y_val,val_prob_xgb)
]).set_index("model").sort_values("PR_AUC",ascending=False)

display(val_results.style.format({
    "PR_AUC":"{:.4f}","ROC_AUC":"{:.4f}",
    "precision":"{:.2%}","recall":"{:.2%}","F1":"{:.4f}"
}))

best_model_name = val_results.index[0]
val_prob_best = val_prob_xgb if best_model_name=="XGBoost" else val_prob_logit
print(f"Selected on validation PR-AUC: {best_model_name}")

fig,ax = plt.subplots(figsize=(8.5,5.2))
for name,prob in [("Logistic Regression",val_prob_logit),("XGBoost",val_prob_xgb)]:
    p,r,_ = precision_recall_curve(y_val,prob)
    ax.plot(r,p,label=f"{name} (AP={average_precision_score(y_val,prob):.3f})")
ax.set_title("Validation precision-recall curves")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.show()

## 7. Cost-sensitive threshold

Illustrative decision economics:
- $5 analyst cost per false alert;
- missed fraud cost = transaction amount + $150 operational impact.

In [ ]:
FALSE_ALERT_COST = 5.0
MISSED_FRAUD_FIXED_COST = 150.0

def decision_cost(y_true,prob,threshold,amounts):
    y = np.asarray(y_true).astype(int)
    pred = (np.asarray(prob)>=threshold).astype(int)
    fp = (pred==1)&(y==0)
    fn = (pred==0)&(y==1)
    return float(fp.sum()*FALSE_ALERT_COST + (np.asarray(amounts)[fn]+MISSED_FRAUD_FIXED_COST).sum())

thresholds = np.linspace(0.005,0.995,300)
costs = np.array([decision_cost(y_val,val_prob_best,t,val_df["amt"].to_numpy()) for t in thresholds])
best_threshold = float(thresholds[np.argmin(costs)])

print(f"Cost-optimised validation threshold: {best_threshold:.3f}")
print(f"Minimum illustrative validation cost: ${costs.min():,.2f}")

fig,ax = plt.subplots(figsize=(9,4.6))
ax.plot(thresholds,costs)
ax.axvline(best_threshold,linestyle="--",label=f"Chosen={best_threshold:.3f}")
ax.set_title("Validation decision cost vs threshold")
ax.set_xlabel("Risk threshold"); ax.set_ylabel("Illustrative cost ($)"); ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.show()

## 8. Final test performance + capture@K

In [ ]:
test_prob_logit = logit.predict_proba(X_test_t)[:,1]
test_prob_xgb = xgb_model.predict_proba(X_test_t)[:,1]
test_prob_best = test_prob_xgb if best_model_name=="XGBoost" else test_prob_logit

test_result = metrics_row(best_model_name,y_test,test_prob_best,best_threshold)
display(pd.DataFrame([test_result]).set_index("model").style.format({
    "PR_AUC":"{:.4f}","ROC_AUC":"{:.4f}",
    "precision":"{:.2%}","recall":"{:.2%}","F1":"{:.4f}"
}))

test_pred = (test_prob_best>=best_threshold).astype(int)
print(classification_report(y_test,test_pred,target_names=["Legitimate","Fraud"],zero_division=0))

cm = confusion_matrix(y_test,test_pred)
fig,ax = plt.subplots(figsize=(5.5,4.5))
im = ax.imshow(cm)
ax.set_title("Test confusion matrix"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_xticks([0,1],["Legit","Fraud"]); ax.set_yticks([0,1],["Legit","Fraud"])
for i in range(2):
    for j in range(2):
        ax.text(j,i,f"{cm[i,j]:,}",ha="center",va="center")
plt.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

def capture_at_k(y_true,score,k_values=(50,100,250,500,1000)):
    ranked = pd.DataFrame({"y":np.asarray(y_true),"score":np.asarray(score)}).sort_values("score",ascending=False)
    total = ranked["y"].sum()
    rows=[]
    for k in k_values:
        k_eff=min(k,len(ranked))
        found=int(ranked.head(k_eff)["y"].sum())
        rows.append({
            "review_capacity":k_eff,"frauds_found":found,
            "precision_at_k":found/k_eff,
            "fraud_capture_rate":found/total if total else np.nan
        })
    return pd.DataFrame(rows)

capture_test = capture_at_k(y_test,test_prob_best)
display(capture_test.style.format({"precision_at_k":"{:.2%}","fraud_capture_rate":"{:.2%}"}))

fig,ax = plt.subplots(figsize=(8,4.6))
ax.plot(capture_test["review_capacity"],capture_test["fraud_capture_rate"]*100,marker="o")
ax.set_title("Fraud captured vs investigator capacity")
ax.set_xlabel("Transactions reviewed"); ax.set_ylabel("Fraud captured (%)"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

test_cost = decision_cost(y_test,test_prob_best,best_threshold,test_df["amt"].to_numpy())
print(f"Illustrative test decision cost: ${test_cost:,.2f}")

## 9. Readable XGBoost explainability

Feature importance and native XGBoost SHAP contributions are mapped back to readable feature names.

In [ ]:
importance = pd.Series(xgb_model.feature_importances_,index=feature_names).sort_values(ascending=False).head(15)
display(importance.to_frame("xgboost_importance"))

fig,ax = plt.subplots(figsize=(8,6))
importance.sort_values().plot(kind="barh",ax=ax)
ax.set_title("Top XGBoost feature importances"); ax.set_xlabel("Importance")
plt.tight_layout(); plt.show()

rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(X_val_t.shape[0],size=min(1500,X_val_t.shape[0]),replace=False)
contrib = xgb_model.get_booster().predict(DMatrix(X_val_t[idx]),pred_contribs=True)
mean_abs_shap = pd.Series(np.abs(contrib[:,:-1]).mean(axis=0),index=feature_names).sort_values(ascending=False).head(15)

display(mean_abs_shap.to_frame("mean_abs_SHAP"))
fig,ax = plt.subplots(figsize=(8,6))
mean_abs_shap.sort_values().plot(kind="barh",ax=ax)
ax.set_title("Global SHAP importance — readable fraud drivers")
ax.set_xlabel("Mean |SHAP contribution|")
plt.tight_layout(); plt.show()

## 10. False-positive audit by merchant category

In [ ]:
audit = test_df[["category","amt","hour","is_fraud"]].copy()
audit["prediction"] = test_pred
audit["false_positive"] = ((audit["prediction"]==1)&(audit["is_fraud"]==0)).astype(int)

segment_audit = audit.groupby("category").agg(
    transactions=("is_fraud","size"),
    legitimate=("is_fraud",lambda s:int((s==0).sum())),
    false_positives=("false_positive","sum")
)
segment_audit["false_positive_rate"] = segment_audit["false_positives"]/segment_audit["legitimate"].replace(0,np.nan)
segment_audit = segment_audit.sort_values("false_positive_rate",ascending=False)

display(segment_audit.style.format({"false_positive_rate":"{:.3%}"}))
fig,ax = plt.subplots(figsize=(10,5))
segment_audit.sort_values("false_positive_rate")["false_positive_rate"].mul(100).plot(kind="barh",ax=ax)
ax.set_title("False-positive rate by merchant category"); ax.set_xlabel("False-positive rate (%)")
plt.tight_layout(); plt.show()

## 11. Analyst investigation queue

In [ ]:
queue = test_df[[
    "trans_date_trans_time","merchant","category","amt","hour","distance_km",
    "customer_txn_count_prior","customer_avg_amount_prior",
    "amount_vs_customer_avg","hours_since_prev_txn","is_fraud"
]].copy()

queue["risk_score"] = test_prob_best
queue["alert"] = (queue["risk_score"]>=best_threshold).astype(int)
rank_pct = queue["risk_score"].rank(method="first",pct=True)
queue["risk_tier"] = pd.cut(rank_pct,bins=[0,0.95,0.99,0.999,1.0],
                            labels=["Low","Medium","High","Critical"],include_lowest=True)
queue = queue.rename(columns={"is_fraud":"actual_fraud"}).sort_values("risk_score",ascending=False)

print("Top 20 transactions for manual investigation:")
display(queue.head(20).style.format({
    "amt":"${:,.2f}","distance_km":"{:,.1f}",
    "customer_avg_amount_prior":"${:,.2f}",
    "amount_vs_customer_avg":"{:.2f}x",
    "hours_since_prev_txn":"{:.1f}","risk_score":"{:.4f}"
}))

queue.head(500).to_csv("fraud_investigation_queue_top500.csv",index=False)
capture_test.to_csv("fraud_capture_at_k.csv",index=False)
segment_audit.to_csv("fraud_false_positive_audit.csv")
print("Operational CSV outputs exported.")

## 12. Executive summary

In [ ]:
k100 = capture_test.loc[capture_test["review_capacity"]==min(100,len(test_df))]
k100 = k100.iloc[0] if len(k100) else capture_test.iloc[0]

print(f"""
EXECUTIVE FRAUD-RISK SUMMARY
----------------------------
Dataset: Sparkov simulated card transactions
Model inputs: human-readable transaction + behavioural features
Selected model: {best_model_name}
Validation-selected threshold: {best_threshold:.3f}

TEST PERFORMANCE
PR-AUC: {test_result['PR_AUC']:.4f}
ROC-AUC: {test_result['ROC_AUC']:.4f}
Precision: {test_result['precision']:.2%}
Recall: {test_result['recall']:.2%}
F1: {test_result['F1']:.4f}
Alerts generated: {test_result['alerts']:,}

OPERATIONS
Top-100 precision: {k100['precision_at_k']:.2%}
Top-100 fraud capture: {k100['fraud_capture_rate']:.2%}
Illustrative decision cost: ${test_cost:,.2f}

WHY THIS VERSION IS STRONGER
The fraud score is explainable in business language: amount, merchant category,
time, geographic distance and prior customer behaviour. Anonymous PCA columns
are gone, sensitive identity fields are excluded, the split is chronological,
threshold selection is isolated from final testing, and scores become a ranked
analyst investigation queue.
""")

### Interview defence

**“I replaced anonymous PCA fraud features with a business-readable transaction dataset. I engineered leakage-safe behavioural signals from prior customer activity, excluded identity and protected demographic fields, used a chronological validation design, selected the model on PR-AUC, tuned the operating threshold against explicit fraud-review costs, audited false positives by merchant segment, and converted risk scores into an analyst-ready investigation queue.”**

### Production extensions
- online feature store for rolling customer behaviour;
- device/IP/merchant graph features;
- probability calibration and delayed-label handling;
- drift monitoring;
- per-segment threshold governance;
- analyst feedback loops;
- champion/challenger deployment;
- alert-volume and latency SLAs.